In [21]:
from hetero_isas.monodromy_lp.invariants import recover_local_equivalence
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, random_unitary
from qiskit.synthesis.two_qubit import TwoQubitWeylDecomposition
from hetero_isas.zz_parallel_drive.bgate import BGate
from weylchamber import g1g2g3

In [22]:
def b_sandwich(target, local_equiv=False):
    # https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.93.020502
    target_decomp = TwoQubitWeylDecomposition(target, fidelity=1.0)
    c1, c2, c3 = target_decomp.a, target_decomp.b, target_decomp.c
    rb1 = 1 - 4 * np.sin(c2) ** 2 * np.cos(c3) ** 2
    b1 = np.arccos(rb1)
    rb2 = np.sqrt(
        np.cos(2 * c2) * np.cos(2 * c3) / (1 - 2 * np.sin(c2) ** 2 * np.cos(c3) ** 2)
    )
    b2 = np.arcsin(rb2)
    temp = QuantumCircuit(2)
    temp.append(BGate(), [0, 1])
    temp.ry(-2 * c1, 0)
    temp.rz(-b2, 1)
    temp.ry(-b1, 1)
    temp.rz(-b2, 1)
    temp.append(BGate(), [0, 1])

    if local_equiv:
        return recover_local_equivalence(target, Operator(temp))
    return temp

In [45]:
target = random_unitary(4)
print(g1g2g3(Operator(target).data))

b_sandwich(target, local_equiv=True)

(np.float64(-0.20069568), np.float64(0.01254125), np.float64(-0.83021365))


(array([[ 0.64138442+0.74151774j, -0.02865485+0.19482393j],
        [ 0.02865485+0.19482393j,  0.64138442-0.74151774j]]),
 array([[ 0.59307876-0.79712371j,  0.0439016 +0.10451807j],
        [-0.0439016 +0.10451807j,  0.59307876+0.79712371j]]),
 array([[ 0.33561915+0.23416376j, -0.87457726-0.26008026j],
        [ 0.87457726-0.26008026j,  0.33561915-0.23416376j]]),
 array([[ 0.98602103-0.02699749j,  0.00635597-0.16429629j],
        [-0.00635597-0.16429629j,  0.98602103+0.02699749j]]),
 0.1990852473423197)